In [1]:
from script.model.pepbridge import PepBridge
from script.model.lora import build_model_with_lora
from script.dataset import PTDataSet, MPDataSet, MPTDataSet
from script.dataprocess import mk_aa_dict, mk_bv_dict
from script.utils import EarlyStopping, model_fn

import pandas as pd 
import numpy as np
import os
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler


In [2]:
aa_dict = mk_aa_dict()
bv_dcit = mk_bv_dict()

In [3]:
model = model_fn(aa_vocab_size=len(aa_dict),
                trbv_vocab_size=len(bv_dcit))

In [4]:
model = build_model_with_lora(model=model, last_n=2, 
                      cfg_seq_pair=((8,16),(4,8)), 
                      dropout=0.1, freeze_base=True,
                      print_trainabel=True)

[Trainable Modules] (581):
  - cdr3_ln
  - chain_id_embedder
  - imm_pred_head.g_s_proj
  - imm_pred_head.g_z_proj
  - imm_pred_head.linear_pair
  - imm_pred_head.linear_seq
  - imm_pred_head.ln_s
  - imm_pred_head.ln_z
  - imm_pred_head.mlp.0
  - imm_pred_head.mlp.3
  - mhc_encoder.embedder.abs_pos_emb
  - mhc_encoder.embedder.mhc_cross_attn
  - mhc_encoder.embedder.mhc_cross_attn.out_proj
  - mhc_encoder.embedder.mhc_poj
  - mhc_encoder.embedder.pair_aa_emb_i
  - mhc_encoder.embedder.pair_aa_emb_j
  - mhc_encoder.embedder.pair_ln
  - mhc_encoder.embedder.rel_pos_emb.embed
  - mhc_encoder.embedder.seq_aa_emb
  - mhc_encoder.embedder.seq_ln
  - mhc_encoder.pair_aware_trunk.pair_aware_trunk.0.ln
  - mhc_encoder.pair_aware_trunk.pair_aware_trunk.0.pair_to_seq.linear
  - mhc_encoder.pair_aware_trunk.pair_aware_trunk.0.pair_to_seq.ln
  - mhc_encoder.pair_aware_trunk.pair_aware_trunk.0.seq_attn.g_proj
  - mhc_encoder.pair_aware_trunk.pair_aware_trunk.0.seq_attn.k_proj
  - mhc_encoder.pair_a

In [5]:
pt_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC_TCR/dataset/pt_train.csv', 
                    header=0, index_col=0)

mp_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC/mp_train.csv', 
                    header=0, index_col=0)
mpt_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC_TCR/dataset/mpt_train.csv', 
                    header=0, index_col=0)
align_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC/random_pMHC_cdr3.csv', 
                    header=0, index_col=0)

In [6]:
imm_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/immuno-genicity/immunogenicity_train.csv', 
                    header=0, index_col=0)
mp_contact_df = pd.read_csv('./data/mp_pdb.csv', 
                    header=0, index_col=0)
pt_contact_df = pd.read_csv('./data/pt_pdb.csv', 
                    header=0, index_col=0)

In [7]:
mp_contact_df

,peptide,MHC,pesudo_seq,pdb_chains
0,YLSPIASPLL,HLA-A02:01,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,5fdw_C_A_pseudo
1,VLHDDLLEA,HLA-A02:01,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,3d25_C_A_pseudo
2,KVAELVHFL,HLA-A02:01,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,3v5d_C_A_pseudo
3,EEFGRAFSF,HLA-B44:03,YYTKYREISTNTYENTAYIRYDDYTWAVLAYLSY,1n2r_C_A_pseudo
4,YLKEPVHGV,HLA-A02:01,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,1i1y_C_A_pseudo
...,...,...,...,...
343,GTSGSPIVNR,HLA-A11:01,YYAMYQENVAQTDVDTLYIIYRDYTWAAQAYRWY,5wkf_C_A_pseudo
344,ELAGIGILTV,HLA-A02:01,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,5nht_P_H_pseudo
345,LTVQVARVY,HLA-B57:03,YYAMYGENMASTYENIAYIVYNYYTWAVLAYLWY,5vwf_C_A_pseudo
346,SLYNTVATL,HLA-A02:01,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,2v2w_C_A_pseudo
